In [0]:
# Databricks Notebook
# 01_bronze_ingestion

import requests
import json

from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    sha2
)

# ============================================================
# Configuration
# ============================================================

api_url = "https://api.transport.nsw.gov.au/v1/carpark/full-list"

api_key = dbutils.secrets.get(
    scope="smart-parking",
    key="tfnsw-api-key"
)

headers = {
    "Authorization": f"apikey {api_key}"
}

SOURCE_SYSTEM = "TfNSW Car Park API"

BRONZE_TABLE = "smart_carparking.bronze_carpark_raw"

# ============================================================
# Extract data from API
# ============================================================

response = requests.get(api_url, headers=headers)

if response.status_code != 200:
    raise Exception(
        f"API request failed. Status: {response.status_code}, Response: {response.text}"
    )

data = response.json()

# ============================================================
# Convert API response into Spark DataFrame
# ============================================================

# Store each API record as a raw JSON string.
# This keeps the Bronze layer close to the original source.
raw_rows = [(json.dumps(record),) for record in data]

bronze_df = spark.createDataFrame(raw_rows, ["raw_json"])

# ============================================================
# Add metadata columns
# ============================================================

bronze_df = (
    bronze_df
    # Timestamp showing when the record was ingested into Databricks
    .withColumn("ingestion_timestamp", current_timestamp())

    # Fixed value identifying the source system
    .withColumn("source_system", lit(SOURCE_SYSTEM))

    # Hash used to identify duplicated raw records
    .withColumn("record_hash", sha2(col("raw_json"), 256))
)

# ============================================================
# Basic data quality filtering
# ============================================================

bronze_df = (
    bronze_df
    # Remove records where raw JSON is null
    .filter(col("raw_json").isNotNull())

    # Remove duplicated records within the same ingestion batch
    .dropDuplicates(["record_hash"])
)

# ============================================================
# Write to Bronze Delta table
# ============================================================

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

# ============================================================
# Display results
# ============================================================

display(bronze_df)

raw_json,ingestion_timestamp,source_system,record_hash
"{""tsn"": ""211420"", ""time"": ""832471554"", ""spots"": ""151"", ""zones"": [{""spots"": ""151"", ""zone_id"": ""1"", ""occupancy"": {""loop"": ""524008"", ""total"": ""151"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""zone_name"": ""SYD372 West Ryde Park&Ride"", ""parent_zone_id"": ""0""}], ""ParkID"": ""1"", ""location"": {""suburb"": ""West Ryde"", ""address"": ""Ryedale Road"", ""latitude"": ""-33.805993"", ""longitude"": ""151.091248""}, ""occupancy"": {""loop"": ""524008"", ""total"": ""151"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""MessageDate"": ""2026-05-19T12:05:54"", ""facility_id"": ""14"", ""facility_name"": ""Park&Ride - West Ryde"", ""tfnsw_facility_id"": ""211420TPR001""}",2026-05-19T02:06:56.622Z,TfNSW Car Park API,008a2fbc3b9f229ac6ea14fe56143175de7c126bce3db521068619125c008ab3
"{""tsn"": ""220910"", ""time"": ""832471309"", ""spots"": ""200"", ""zones"": [{""spots"": ""200"", ""zone_id"": ""1"", ""occupancy"": {""loop"": ""30939"", ""total"": ""129"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""zone_name"": ""SYD409 Beverly Hills Park&Ride"", ""parent_zone_id"": ""0""}], ""ParkID"": ""1"", ""location"": {""suburb"": ""Beverly Hills"", ""address"": ""2-2A Edgbaston Road"", ""latitude"": ""-33.949744"", ""longitude"": ""151.0801""}, ""occupancy"": {""loop"": ""30939"", ""total"": ""129"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""MessageDate"": ""2026-05-19T12:01:49"", ""facility_id"": ""35"", ""facility_name"": ""Park&Ride - Beverly Hills"", ""tfnsw_facility_id"": ""220910TPR001""}",2026-05-19T02:06:56.622Z,TfNSW Car Park API,01c10e8d37e31da7820a7bca073f8f7e8a28202a564b810c398297f62fbe8d6d
"{""tsn"": ""256020"", ""time"": ""832471242"", ""spots"": ""113"", ""zones"": [{""spots"": ""113"", ""zone_id"": ""1"", ""occupancy"": {""loop"": ""442335"", ""total"": ""113"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""zone_name"": ""SYD368C HSS Park&Ride"", ""parent_zone_id"": ""0""}], ""ParkID"": ""1"", ""location"": {""suburb"": ""Campbelltown"", ""address"": ""Hurley Street"", ""latitude"": ""-34.065798"", ""longitude"": ""150.812432""}, ""occupancy"": {""loop"": ""442335"", ""total"": ""113"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""MessageDate"": ""2026-05-19T12:00:42"", ""facility_id"": ""20"", ""facility_name"": ""Park&Ride - Campbelltown Hurley St"", ""tfnsw_facility_id"": ""256020TPR002""}",2026-05-19T02:06:56.622Z,TfNSW Car Park API,03bbd932cf0e9aa4d31e9d2d5c6f5724e365230f1fb3fd3cc951bec238ea1672
"{""tsn"": ""275010"", ""time"": ""832471534"", ""spots"": ""1129"", ""zones"": [{""spots"": ""1129"", ""zone_id"": ""1"", ""occupancy"": {""loop"": ""1406377"", ""total"": ""827"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""zone_name"": ""SYD369B Penrith MSC Park&Ride"", ""parent_zone_id"": ""0""}], ""ParkID"": ""1"", ""location"": {""suburb"": ""Penrith"", ""address"": ""Combewood Avenue"", ""latitude"": ""-33.748452"", ""longitude"": ""150.695171""}, ""occupancy"": {""loop"": ""1406377"", ""total"": ""827"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""MessageDate"": ""2026-05-19T12:05:34"", ""facility_id"": ""22"", ""facility_name"": ""Park&Ride - Penrith (multi-level)"", ""tfnsw_facility_id"": ""275010TPR002""}",2026-05-19T02:06:56.622Z,TfNSW Car Park API,05b935adc33b66c6ca9d28694b3bf0dba13e6b1c7d01369080740bb2ee2e7f69
"{""tsn"": ""2101130"", ""time"": ""832471142"", ""spots"": ""46"", ""zones"": [{""spots"": ""46"", ""zone_id"": ""1"", ""occupancy"": {""loop"": ""154061"", ""total"": ""46"", ""monthlies"": null, ""open_gate"": null, ""transients"": null}, ""zone_name"": ""SYD312 Narrabeen Park&Ride"", ""parent_zone_id"": ""0""}], ""ParkID"": ""1"", ""location"": {""suburb"": ""Narrabeen"", ""address"": ""Pittwater Road"", ""latitude"": ""-33.714364"", ""longitude